# C5 — Sharpening the detection signal without retraining: residual band-pass + classical baselines

`C4` established that detection is carried by the **structural** residual, that
normalising by a sigma (learned or adaptive) does nothing, and that the architecture
axis is saturated (`final-comments.md`). Two cheap, training-free levers remain:

1. **Multi-scale band-pass of the residual.** Underdrawing strokes, craquelure
   (high-frequency), and substrate/lighting shifts (low-frequency) live in distinct
   spatial-frequency bands. A difference-of-Gaussians band-pass tuned to stroke width
   should suppress both the craquelure noise floor and the slow substrate trends,
   leaving the stroke band. Applied to `structural delta` (the current best signal)
   and to the cheap `linreg residual`.

2. **Classical per-image decompositions.** What conservators actually use — no model,
   no train/test transfer problem, per image:
   - `linreg residual`: per-image OLS of `IR ~ [R, G, B, 1]`, residual `|IR - pred|` —
     a parameter-free linear `μ`, the natural baseline for the deep `μ`.
   - `PCA` / `FastICA` on the standardised `[R, G, B, IR]` stack — the underdrawing
     usually surfaces as one component; we score every component against the mask and
     report the best (an operator would pick it visually).

Both are scored against `structural delta (deep μ)` on AUROC (`GT01`-`GT03`) and stroke
coherence, and shown side by side in a **visual gallery** for perceptual assessment.

**Memory.** `IMAGE_STEMS` (§1) lists every `data/test/` image with the non-GT ones
commented out — enable them one or a few at a time. AUROC needs a mask so it only ever
uses the GT images; coherence and the gallery use whatever is enabled.

_Nothing here retrains anything._

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports. `scipy.ndimage` for the band-pass, `sklearn.decomposition` for PCA/ICA, the `C0`/`C4` toolkit for everything else.

In [ ]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.decomposition import PCA, FastICA

from scripts.config import settings
from scripts.dataset import pad_to_multiple
from scripts.delta_analysis import analyze_delta
from scripts.detection import evaluate_detection
from scripts.stroke_stats import stroke_coherence
from scripts.trainer import load_model
from scripts.visualization import plot_gt_signal_gallery
from scripts.visualization_nll import plot_signal_gallery

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Test images — enable what you need

`IMAGE_STEMS` drives everything. The three GT images are on; the rest are commented
out. Uncomment a few at a time — full-resolution paintings are large and PCA/ICA
allocate `H*W x 4` float64 arrays per image.

In [ ]:
IMAGE_STEMS = [
    "GT01",
    "GT02",
    "GT03",
    # --- non-GT images (no mask): uncomment as needed, ideally one at a time ---
    # "bridge",
    # "case",
    # "face",
    # "green",
    # "modern",
    # "modern2",
    # "total",
]

TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR = project_root / "data" / "test" / "ir"
ANNOTATIONS_DIR = project_root / "data" / "test" / "annotations"


def _find(stem: str, folder: Path) -> Path | None:
    for ext in (".jpg", ".png", ".jpeg", ".tif", ".tiff"):
        if (folder / f"{stem}{ext}").exists():
            return folder / f"{stem}{ext}"
    return None


image_pairs = []
for stem in IMAGE_STEMS:
    rgb_p, ir_p = _find(stem, TEST_RGB_DIR), _find(stem, TEST_IR_DIR)
    if rgb_p and ir_p:
        image_pairs.append((stem, rgb_p, ir_p))
    else:
        print(f"[skip] {stem}: rgb={rgb_p} ir={ir_p}")

gt_stems = {
    p.name.removesuffix("_Map.png") for p in ANNOTATIONS_DIR.glob("*_Map.png")
} & {s for s, _, _ in image_pairs}

print(f"enabled: {[s for s, _, _ in image_pairs]}")
print(f"with a GT mask: {sorted(gt_stems)}")
if not image_pairs:
    raise RuntimeError("IMAGE_STEMS resolved to nothing.")

## 2. The predictor for `μ`

One deterministic checkpoint supplies `μ`. Default `attention_unet` — `C4`'s k-fold
pick and the architecture whose `structural delta` scored best. Swap `MU_ARCH` for any
`models/deterministic/` key.

In [ ]:
MU_ARCH = "attention_unet"
mu_model = load_model(MU_ARCH, model_dir=settings.MODELS_DIR / "deterministic")
print(f"loaded {MU_ARCH}")

## 3. Signal construction

`build_signals(stem, rgb, ir)` returns every candidate signal for one image:

- `raw delta`, `structural delta` — the deep-`μ` baselines (`structural delta` is the
  reference every other signal is compared against).
- `bp[s_lo-s_hi](structural delta)` / `bp[...](linreg residual)` — DoG band-pass,
  positive part, one per band in `BANDS`.
- `linreg residual` — per-image OLS `IR ~ [R,G,B,1]`.
- `pca c{k}` / `ica c{k}` — the k standardised-`[R,G,B,IR]` components (median-centred,
  abs).

In [ ]:
BANDS = [(1, 6), (2, 12), (3, 20), (1, 30)]  # (sigma_low, sigma_high) in pixels
N_COMPONENTS = 4


def bandpass(x: np.ndarray, s_lo: float, s_hi: float) -> np.ndarray:
    """Difference-of-Gaussians band-pass; keeps structure with scale ~[s_lo, s_hi] px.

    Positive part only: an underdrawing stroke reads brighter than its local
    background in the residual, so ``blur_fine - blur_coarse >= 0`` there.
    """
    fine = gaussian_filter(x.astype(np.float32), s_lo, mode="reflect")
    coarse = gaussian_filter(x.astype(np.float32), s_hi, mode="reflect")
    return np.clip(fine - coarse, 0.0, None)


def linreg_residual(rgb: np.ndarray, ir: np.ndarray) -> np.ndarray:
    """|IR - OLS(IR ~ [R,G,B,1])| — a parameter-free per-image linear predictor."""
    h, w = ir.shape
    A = np.column_stack([rgb.reshape(-1, 3), np.ones(h * w, np.float32)])
    coef, *_ = np.linalg.lstsq(A, ir.reshape(-1), rcond=None)
    return np.abs(ir - (A @ coef).reshape(h, w))


def decomposition(rgb: np.ndarray, ir: np.ndarray, method: str, n: int) -> dict:
    """PCA / FastICA on the standardised [R,G,B,IR] stack; each component -> a map."""
    h, w = ir.shape
    X = np.column_stack([rgb.reshape(-1, 3), ir.reshape(-1, 1)]).astype(np.float64)
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)
    if method == "pca":
        comps = PCA(n_components=n, random_state=0).fit_transform(X)
    else:
        comps = FastICA(n_components=n, random_state=0, max_iter=800, whiten="unit-variance").fit_transform(X)
    out = {}
    for k in range(comps.shape[1]):
        c = comps[:, k].reshape(h, w)
        out[f"{method} c{k}"] = np.abs(c - np.median(c))
    return out


def build_signals(rgb: np.ndarray, ir: np.ndarray) -> dict[str, np.ndarray]:
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    h, w = ir.shape
    mu = mu_model.predict(padded[tf.newaxis, ...], verbose=0)[0, :h, :w, 0]

    d = analyze_delta(ir, mu)
    lin = linreg_residual(rgb, ir)

    sig = {
        "raw delta": d.raw_delta,
        "structural delta": d.structural_delta,     # <- reference
        "linreg residual": lin,
    }
    for s_lo, s_hi in BANDS:
        sig[f"bp[{s_lo}-{s_hi}] struct"] = bandpass(d.structural_delta, s_lo, s_hi)
        sig[f"bp[{s_lo}-{s_hi}] linreg"] = bandpass(lin, s_lo, s_hi)
    sig.update(decomposition(rgb, ir, "pca", N_COMPONENTS))
    sig.update(decomposition(rgb, ir, "ica", N_COMPONENTS))
    return sig


REFERENCE = "structural delta"

## 4. Score every signal on every enabled image

AUROC only for images with a mask; coherence for all. One image at a time, arrays
dropped before the next.

In [ ]:
def load_pair(rgb_p: Path, ir_p: Path):
    rgb = np.array(Image.open(rgb_p).convert("RGB"), np.float32) / 255.0
    ir = np.array(Image.open(ir_p).convert("L"), np.float32) / 255.0
    return rgb, ir


def load_mask(stem: str) -> np.ndarray:
    return np.array(Image.open(ANNOTATIONS_DIR / f"{stem}_Map.png").convert("L")) > 127


auroc: dict[str, dict[str, float]] = {}
ap: dict[str, dict[str, float]] = {}
coh: dict[str, list[float]] = {}
gallery_cache: dict[str, tuple] = {}

for stem, rgb_p, ir_p in image_pairs:
    rgb, ir = load_pair(rgb_p, ir_p)
    signals = build_signals(rgb, ir)
    mask = load_mask(stem) if stem in gt_stems else None

    for name, s in signals.items():
        coh.setdefault(name, []).append(stroke_coherence(s).coherence)
        if mask is not None:
            r = evaluate_detection(s, mask)
            auroc.setdefault(name, {})[stem] = r.auroc
            ap.setdefault(name, {})[stem] = r.average_precision

    gallery_cache[stem] = (rgb, ir, mask, signals)
    print(f"  {stem}: {len(signals)} signals scored"
          + (f" (AUROC vs mask)" if mask is not None else " (coherence only)"))

signal_names = list(next(iter(gallery_cache.values()))[3].keys())
print(f"\n{len(signal_names)} signals, {len(image_pairs)} images")

## 5. Detection AUROC

Mean over the enabled GT images, sorted best-first. `Δ` is against
`structural delta (deep μ)`. With `n = 3` (default) this is indicative, not
conclusive — read §7's gallery alongside.

In [ ]:
if auroc:
    ref = float(np.mean(list(auroc[REFERENCE].values())))
    rows = sorted(
        ((name, float(np.mean(list(v.values())))) for name, v in auroc.items()),
        key=lambda t: -t[1],
    )
    print(f"reference '{REFERENCE}' = {ref:.4f}   (mean over {sorted(gt_stems)})\n")
    print(f"  {'signal':<24}{'AUROC':>9}{'Δ vs ref':>12}")
    print("  " + "-" * 45)
    for name, v in rows:
        mark = "  <-- ref" if name == REFERENCE else f"{v - ref:+.4f}".rjust(12)
        print(f"  {name:<24}{v:>9.4f}{mark}")
else:
    print("No GT mask among the enabled images — AUROC skipped.")

### Per-image AUROC — the top signals

In [ ]:
if auroc:
    stems = sorted(gt_stems)
    top = [n for n, _ in rows[:12]]
    print(f"  {'signal':<24}" + "".join(s.rjust(9) for s in stems))
    print("  " + "-" * (24 + 9 * len(stems)))
    for name in top:
        print(f"  {name:<24}" + "".join(f"{auroc[name][s]:.3f}".rjust(9) for s in stems))
else:
    print("skipped")

## 6. Stroke coherence

Mean over all enabled images. A band-pass that isolated the stroke band should read
*higher* here than the un-filtered `structural delta`.

In [ ]:
ref_c = float(np.mean(coh[REFERENCE]))
rows_c = sorted(((n, float(np.mean(v)), float(np.std(v))) for n, v in coh.items()), key=lambda t: -t[1])
print(f"reference '{REFERENCE}' = {ref_c:.4f}   (mean over {[s for s, _, _ in image_pairs]})\n")
print(f"  {'signal':<24}{'coherence':>12}{'Δ vs ref':>12}")
print("  " + "-" * 48)
for name, m, sd in rows_c:
    mark = "  <-- ref" if name == REFERENCE else f"{m - ref_c:+.4f}".rjust(12)
    print(f"  {name:<24}{f'{m:.3f}±{sd:.3f}':>12}{mark}")

## 7. Visual comparison — perceptual assessment

For each enabled image: the context panels (RGB, real IR, mask if present) followed by
the signal maps, each on its own percentile scale. `PANEL_SIGNALS` picks which signals
to show — keep it short or the figure gets unreadable.

In [ ]:
PANEL_SIGNALS = [
    "structural delta",
    "linreg residual",
    "bp[2-12] struct",
    "bp[3-20] struct",
    "bp[2-12] linreg",
    "pca c3",
    "ica c0",
    "ica c3",
]

for stem, (rgb, ir, mask, signals) in gallery_cache.items():
    panels = {k: signals[k] for k in PANEL_SIGNALS if k in signals}
    if mask is not None:
        fig = plot_gt_signal_gallery(rgb, ir, mask, panels, title=f"C5 — {stem} ({MU_ARCH} μ)")
    else:
        fig = plot_signal_gallery(ir, panels, title=f"C5 — {stem} ({MU_ARCH} μ)")
    plt.show()
    plt.close(fig)

### All PCA / ICA components for one image

To see which component (if any) actually carries the underdrawing — pick the stem.

In [ ]:
INSPECT_STEM = sorted(gt_stems)[0] if gt_stems else image_pairs[0][0]
rgb, ir, mask, signals = gallery_cache[INSPECT_STEM]
comps = {k: v for k, v in signals.items() if k.startswith(("pca c", "ica c"))}
if mask is not None:
    fig = plot_gt_signal_gallery(rgb, ir, mask, comps, title=f"C5 — {INSPECT_STEM} — all components")
else:
    fig = plot_signal_gallery(ir, comps, title=f"C5 — {INSPECT_STEM} — all components")
plt.show()
plt.close(fig)

## 8. Conclusion

_Fill in after running._

- **Band-pass on `structural delta`** — does any band beat the un-filtered reference on
  AUROC and/or coherence? Which `(s_lo, s_hi)`? …
- **Band-pass on `linreg residual`** — does the cheap linear predictor + band-pass get
  close to the deep `μ`? …
- **PCA / ICA** — does any single component reach the deep signal's AUROC? Is it the
  same component across `GT01`-`GT03` (→ usable) or a different one each time (→ not)? …
- **Perceptual (§7)** — is the underdrawing visibly clearer in any of the new signals
  than in `structural delta`? …

**Decision:**
- If a band-pass clearly wins → adopt `bp[...] structural delta` as the primary signal
  (no retraining, drop-in).
- If a classical decomposition matches the deep signal → report it as an interpretable,
  transfer-free alternative in the write-up.
- If nothing beats `structural delta` → confirms `final-comments.md`'s framing: the
  ceiling is data/acquisition, not signal processing. Close on `structural delta` +
  `structural z`.